# 05 — Seasonality Analysis

Day-of-week effects, day-of-month patterns, and volatility seasonality in crypto markets.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


In [ ]:
from statarb.signals.seasonality import SeasonalitySignals
seas = SeasonalitySignals()
mean_ret = returns.mean(axis=1).rename("universe_mean")


## Day-of-Week Effects

In [ ]:
# Average return by day of week
dow_ret = mean_ret.copy()
dow_ret.index = pd.DatetimeIndex(dow_ret.index)
dow_df = pd.DataFrame({
    "return": dow_ret.values,
    "dow":    dow_ret.index.dayofweek,
    "dow_name": dow_ret.index.day_name(),
})

dow_avg = dow_df.groupby("dow").agg(
    mean_return=("return", "mean"),
    std_return=("return", "std"),
    count=("return", "count"),
).assign(tstat=lambda d: d["mean_return"] / (d["std_return"] / d["count"].pow(0.5)))

labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["green" if v > 0 else "red" for v in dow_avg["mean_return"].values]
ax.bar(range(len(dow_avg)), dow_avg["mean_return"].values * 100, color=colors, alpha=0.8)
ax.set_xticks(range(len(dow_avg)))
ax.set_xticklabels(labels[:len(dow_avg)])
ax.set_title("Average Universe Return by Day of Week", fontweight="bold")
ax.set_ylabel("Mean Daily Return (%)")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()
print(dow_avg[["mean_return", "tstat"]].round(4).to_string())


## Month-of-Year Effects

In [ ]:
month_df = pd.DataFrame({
    "return": mean_ret.values,
    "month":  pd.DatetimeIndex(mean_ret.index).month,
})
month_avg = month_df.groupby("month")["return"].mean()
month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["green" if v > 0 else "red" for v in month_avg.values]
ax.bar(range(1, len(month_avg)+1), month_avg.values * 100, color=colors, alpha=0.8)
ax.set_xticks(range(1, len(month_avg)+1))
ax.set_xticklabels(month_labels[:len(month_avg)], fontsize=9)
ax.set_title("Average Universe Return by Month", fontweight="bold")
ax.set_ylabel("Mean Daily Return (%)")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()


## Volatility Seasonality

In [ ]:
realized_vol = fe.realized_volatility(returns, window=5)
avg_vol = realized_vol.mean(axis=1)

vol_df = pd.DataFrame({
    "vol":   avg_vol.values,
    "dow":   pd.DatetimeIndex(avg_vol.index).dayofweek,
    "month": pd.DatetimeIndex(avg_vol.index).month,
}).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
vol_by_dow = vol_df.groupby("dow")["vol"].mean()
axes[0].bar(range(len(vol_by_dow)), vol_by_dow.values * 100, color="steelblue", alpha=0.8)
axes[0].set_xticks(range(len(vol_by_dow)))
axes[0].set_xticklabels(labels[:len(vol_by_dow)])
axes[0].set_title("Avg 5-Day Vol by Day of Week (%)", fontweight="bold")

vol_by_month = vol_df.groupby("month")["vol"].mean()
axes[1].bar(range(1, len(vol_by_month)+1), vol_by_month.values * 100,
            color="darkorange", alpha=0.8)
axes[1].set_xticks(range(1, len(vol_by_month)+1))
axes[1].set_xticklabels(month_labels[:len(vol_by_month)], fontsize=8)
axes[1].set_title("Avg 5-Day Vol by Month (%)", fontweight="bold")
plt.tight_layout()
plt.show()


## Seasonality Signal Backtest

In [ ]:
from experiments._utils import backtest_signals
from statarb.backtest.execution import ExecutionModel

try:
    sigs = {
        "dow_signal":    seas.day_of_week_signal(returns),
        "month_signal":  seas.month_of_year_signal(returns),
    }
    ranked_sigs = {k: fe.cross_sectional_rank(v) for k, v in sigs.items()}
    res = backtest_signals(ranked_sigs, returns, ExecutionModel(), 365)
    print(res[["sharpe", "annualized_return", "max_drawdown"]].round(3).to_string())
except Exception as e:
    print(f"Seasonality signals unavailable: {e}")
